----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Discovery layer - Instruments - Futures
This notebook covers the public `Futures` class. It discovers futures contracts whose expiry dates fall within a requested range, including expired contracts.
**Prerequisites:** a configured session with access to Search and, for underlying-identifier resolution, fundamental and reference data. Keep credentials in the library configuration. Search availability and listed contracts can change; no fixed result count or RIC is assumed. These examples discover contracts, not prices, and do not place orders.

#### Learn more
Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials.

#### Getting Help and Support
For API questions, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import lseg.data as ld
from lseg.data.discovery import Futures

## Open the data session
`open_session()` uses the default session from `lseg-data.config.json`.

In [ ]:
ld.open_session()

## Discover contracts from a RIC root
`Futures(universe, start, end).get_data()` returns a DataFrame sorted by expiry date. The date range filters **contract expiry**, not trade dates or a historical price window. Use `YYYY-MM-DD` dates.

The tests use `ES` for the range below. Typical columns include `RIC`, `RicRoot`, `ExchangeCode`, `ExpiryDate` and `UnderlyingQuoteRIC`.

In [ ]:
contracts = Futures(
    universe="ES",
    start="2024-12-01",
    end="2025-02-10",
).get_data()
contracts

## Resolve other underlying identifier types
The test also accepts an organization PermID (`27602`), an equity RIC (`MSFT.OQ`) and an index RIC (`.SPX`). Each call resolves one underlying; this is not a multi-instrument batch API. The optional list input is intended to contain a single instrument, so do not pass several underlyings in one list.

In [ ]:
for underlying in ("27602", "MSFT.OQ", ".SPX"):
    resolved = Futures(
        universe=underlying,
        start="2024-12-01",
        end="2025-02-10",
    ).get_data()
    print("Underlying:", underlying)
    display(resolved)

## Discover an expired contract
The exact-match test uses root `MSTF` for December 2023 and expects `MSTFZ3^2`. Keep the expired suffix returned by the service rather than attempting to reconstruct it. This notebook displays the actual result without hard-coding an assertion about its continued availability.

In [ ]:
expired = Futures(universe="MSTF", start="2023-12-01", end="2023-12-31").get_data()
expired

## Discover December 2026 contracts
The non-expired-contract test uses this range and expects `MSTFZ26` when the contract is still live. Whether the range is future or historical depends on when the notebook is run; returned RIC suffixes can change once contracts expire. An empty DataFrame means no matching contract was found, not that a request error should be ignored.

In [ ]:
december_2026 = Futures(universe="MSTF", start="2026-12-01", end="2026-12-31").get_data()
display(december_2026)
if december_2026.empty:
    print("No matching December 2026 contracts were found.")

## Close the session
Run this cell even if an earlier request fails. No server-side resources were created.

In [ ]:
ld.close_session()